# Little Syria, Dispersed: Arab Ancestry and NYC Landmarks
Assignment 2: Two Datasets, a Site, a Map. This follows the structure of Tutorial 2 (Mapping Where).

**Datasets**
1. **People Reporting Ancestry (Table B04006)**, U.S. Census Bureau, ACS 2024 5-Year Estimates, by census tract. Joined to **2020 Census Tracts** from the NYC Department of City Planning (NYC Open Data).
2. **Individual Landmark Sites**, NYC Landmarks Preservation Commission (NYC Open Data).

**Projection:** NAD83 / New York Long Island (ftUS), **EPSG:2263**.

**How to use this notebook:** run each grey code box from top to bottom, one at a time, by clicking the ▶ button on its left. Wait for each one to finish (a green ✓ appears) before running the next.

## 1. Setup
Load the toolkits. This is the same idea as the tutorial's first cell:
- **pandas** handles tables.
- **geopandas** handles tables that have shapes.
- **altair** draws maps and charts.

In [ ]:
from google.colab import files

import pandas as pd
import geopandas as gpd
import altair as alt

Install the tool that lets Altair save the map as a PNG. You only need this once per session.

In [ ]:
!pip install vl-convert-python

## 2. Upload the files
Click **Choose Files** and select all three files at once:
- `ACSDT5Y2024.B04006_....zip`
- `2020_Census_Tracts_....geojson`
- `Individual_Landmark_Sites_....csv`

**Skip this cell if you already uploaded them in this session.** Check the 📁 folder icon on the left: if the files are listed there, they're already uploaded.

In [ ]:
uploaded = files.upload()

The census data arrives as a zip, so unzip it. **If your zip has a different name, change the name after `unzip -o` to match the file in the 📁 panel.**

In [ ]:
!unzip -o /content/ACSDT5Y2024.B04006_*.zip

## 3. Census tracts (the shapes)
Load the tract boundaries.
- `.crs` shows the coordinate system the file came in, which is EPSG:4326 (latitude/longitude).
- `.to_crs(epsg=2263)` re-projects it into NYC's State Plane system, measured in feet.

**If this cell errors with "file not found":** check the exact file name in the 📁 panel and fix it in the code. Uploading twice adds " (1)" to the name.

In [ ]:
gdf_Tracts = gpd.read_file('/content/2020_Census_Tracts_20260922 (1).geojson')
print('Original CRS:', gdf_Tracts.crs)

gdf_Tracts = gdf_Tracts.to_crs(epsg=2263)
print('New CRS:', gdf_Tracts.crs)

gdf_Tracts

A quick look at the empty tracts, like the tutorial's block map. In EPSG:2263 we use `type='identity'`, which means "draw the coordinates as they are" instead of Mercator.

In [ ]:
alt.data_transformers.disable_max_rows()

chart_Tracts = alt.Chart(gdf_Tracts).mark_geoshape(
    stroke='black',
    strokeWidth=0.2
).encode(
    color=alt.value('white')
).project(
    type='identity', reflectY=True
).properties(
    width=700,
    height=700
)

chart_Tracts

## 4. Census ancestry (the numbers)
The census CSV has two header rows: codes, then descriptions. `skiprows=[1]` drops the second one.

The column we want is **`B04006_006E`**: the estimated number of people reporting **Arab** ancestry. This total covers Egyptian, Iraqi, Jordanian, Lebanese, Moroccan, Palestinian, Syrian, Arab and Other Arab. The file covers all of New York State. The NYC tracts are picked out in the join below.

In [ ]:
df_Ancestry = pd.read_csv('/content/ACSDT5Y2024.B04006-Data.csv', skiprows=[1])

df_Ancestry[['GEO_ID', 'NAME', 'B04006_006E']]

The census `GEO_ID` looks like `1400000US36061000700`, but the tract file's `geoid` is only the last 11 digits (`36061000700`). So we cut the census ID down to 11 digits, so the two files share a column to match on.

In [ ]:
df_Arab = df_Ancestry[['GEO_ID', 'B04006_006E']].copy()
df_Arab['geoid'] = df_Arab['GEO_ID'].str[-11:]
df_Arab['arab_ancestry'] = pd.to_numeric(df_Arab['B04006_006E'], errors='coerce')
df_Arab = df_Arab[['geoid', 'arab_ancestry']]

df_Arab

## 5. Attribute join: numbers onto shapes
The tutorial joined by location. This first join is by a **shared ID** instead (`geoid`). Tracts outside NYC fall away because they aren't in the tract file.

In [ ]:
gdf_ArabTracts = gdf_Tracts.merge(df_Arab, on='geoid', how='left')
gdf_ArabTracts['arab_ancestry'] = gdf_ArabTracts['arab_ancestry'].fillna(0)

gdf_ArabTracts.sort_values('arab_ancestry', ascending=False)[['ntaname', 'boroname', 'arab_ancestry']].head(10)

## 6. Landmarks (the second dataset)
The landmarks CSV stores each landmark's shape as text in the `the_geom` column. That shape is already in EPSG:2263, feet.
- `from_wkt` turns that text into real shapes.
- `.centroid` turns each lot into a single point so it can be drawn as a dot.

In [ ]:
df_Landmarks = pd.read_csv('/content/Individual_Landmark_Sites_20260922.csv')

gdf_Landmarks = gpd.GeoDataFrame(
    df_Landmarks,
    geometry=gpd.GeoSeries.from_wkt(df_Landmarks['the_geom']),
    crs='EPSG:2263'
)
gdf_Landmarks['geometry'] = gdf_Landmarks.geometry.centroid

print(len(gdf_Landmarks), 'landmarks')
gdf_Landmarks[['LM_NAME', 'DESIG_ADDRESS', 'DESIG_DATE', 'Borough']]

Find the one landmark from Little Syria on our site:

In [ ]:
gdf_Syrian = gdf_Landmarks[gdf_Landmarks['LM_NAME'].str.contains('Syrian')]

gdf_Syrian[['LM_NAME', 'DESIG_ADDRESS', 'DESIG_DATE']]

## 7. Spatial join: count landmarks in each tract
This step is the same as the tutorial's `sjoin`. Each landmark point is matched to the tract that **contains** it, then we count the points per tract.

In [ ]:
gdf_joined = gpd.sjoin(gdf_ArabTracts, gdf_Landmarks, how='right', predicate='contains')

df_counts = gdf_joined.groupby('geoid').agg(
    landmark_count=('LM_NAME', 'count')
).reset_index()

gdf_ArabTracts = gdf_ArabTracts.merge(df_counts, on='geoid', how='left')
gdf_ArabTracts['landmark_count'] = gdf_ArabTracts['landmark_count'].fillna(0)

gdf_ArabTracts[['ntaname', 'arab_ancestry', 'landmark_count']]

A number worth stating in your description: how many landmarks sit in the 50 tracts with the most residents of Arab ancestry, compared with the whole city.

In [ ]:
top50 = gdf_ArabTracts.sort_values('arab_ancestry', ascending=False).head(50)

print('Landmarks in the 50 tracts with the most Arab ancestry:', int(top50['landmark_count'].sum()))
print('Landmarks in all of NYC:', int(gdf_ArabTracts['landmark_count'].sum()))

## 8. Prepare the layers for Altair
- Simplify the tract outlines slightly (by 50 ft) so the map draws faster.
- Pull x/y numbers out of the landmark points so Altair can place them.

In [ ]:
gdf_ArabTracts['geometry'] = gdf_ArabTracts.geometry.simplify(50)

gdf_Landmarks['x'] = gdf_Landmarks.geometry.x
gdf_Landmarks['y'] = gdf_Landmarks.geometry.y
gdf_Syrian = gdf_Landmarks[gdf_Landmarks['LM_NAME'].str.contains('Syrian')]

df_LandmarkPts = pd.DataFrame(gdf_Landmarks[['x', 'y', 'LM_NAME', 'DESIG_DATE']])
df_SyrianPt = pd.DataFrame(gdf_Syrian[['x', 'y', 'LM_NAME', 'DESIG_DATE']])

## 9. Choropleth: Arab ancestry by tract
This is quantitative symbology, like the tutorial's tree-diameter map. The class breaks (1, 50, 150, 300, 600) are a choice, and changing them changes the argument the map makes. Try your own.

In [ ]:
chart_Arab = alt.Chart(gdf_ArabTracts).mark_geoshape(
    stroke='white',
    strokeWidth=0.2
).encode(
    color=alt.Color('arab_ancestry:Q',
                    scale=alt.Scale(type='threshold',
                                    domain=[1, 50, 150, 300, 600],
                                    range=['#f2f0ec', '#fde0c5', '#fbb07a', '#f07c3a', '#c9471a', '#7f2704']),
                    legend=alt.Legend(title='People reporting Arab ancestry')),
    tooltip=['ntaname:N', 'arab_ancestry:Q', 'landmark_count:Q']
).project(
    type='identity', reflectY=True
).properties(
    width=1100,
    height=620
)

chart_Arab

## 10. Landmark points, and the church on our site

In [ ]:
chart_Landmarks = alt.Chart(df_LandmarkPts).mark_circle(
    size=6, color='#1d3557', opacity=0.5
).encode(
    longitude='x:Q',
    latitude='y:Q',
    tooltip=['LM_NAME:N', 'DESIG_DATE:N']
)

chart_Syrian = alt.Chart(df_SyrianPt).mark_circle(
    size=250, color='red', stroke='black', strokeWidth=1
).encode(
    longitude='x:Q',
    latitude='y:Q',
    tooltip=['LM_NAME:N', 'DESIG_DATE:N']
)

chart_Label = alt.Chart(df_SyrianPt).mark_text(
    align='right', dx=-10, fontSize=12, fontWeight='bold'
).encode(
    longitude='x:Q',
    latitude='y:Q',
    text=alt.value("St. George's Syrian Catholic Church (Little Syria)")
)

chart_Arab + chart_Landmarks + chart_Syrian + chart_Label

## 11. Scale bar and north arrow
EPSG:2263 is measured in feet, so a 2-mile scale bar is simply a line 10,560 ft long.

In [ ]:
minx, miny, maxx, maxy = gdf_ArabTracts.total_bounds
bx, by = maxx - 40000, miny + 8000

chart_Scale = alt.Chart(pd.DataFrame({'x': [bx, bx + 10560], 'y': [by, by]})).mark_line(
    color='black', strokeWidth=3
).encode(longitude='x:Q', latitude='y:Q')

chart_ScaleText = alt.Chart(pd.DataFrame({'x': [bx + 5280], 'y': [by]})).mark_text(dy=-10).encode(
    longitude='x:Q', latitude='y:Q', text=alt.value('2 miles'))

chart_North = alt.Chart(pd.DataFrame({'x': [maxx - 10000], 'y': [maxy - 10000]})).mark_text(
    fontSize=22, fontWeight='bold'
).encode(longitude='x:Q', latitude='y:Q', text=alt.value('↑ N'))

## 12. Final map: title, sources, projection, name
**Replace `YOUR NAME` with your name, and write your own title** (the course requires the words to be yours). Then run the cell.

In [ ]:
Final_Chart = (
    chart_Arab + chart_Landmarks + chart_Syrian + chart_Label
    + chart_Scale + chart_ScaleText + chart_North
).properties(
    title=alt.Title(
        'YOUR TITLE HERE',
        subtitle=[
            'Arab ancestry by census tract (ACS 2020–2024) and NYC individual landmarks. Projection: NAD83 / New York Long Island, EPSG:2263.',
            'Sources: U.S. Census Bureau, ACS 5-Year Table B04006; NYC Landmarks Preservation Commission, Individual Landmark Sites (NYC Open Data). Map by YOUR NAME.'
        ]
    )
).configure_view(stroke=None)

Final_Chart

## 13. Save and download the PNG
This saves the map as `map.png` and downloads it to your computer. Finish the 16:9 layout in Illustrator, Figma or similar if you want, and apply your precedent's graphic style there or by changing colors above.

In [ ]:
Final_Chart.save('/content/map.png', scale_factor=2)
files.download('/content/map.png')